# Video indexing — Colab L4 runs

Runtime: **L4 GPU** (Runtime → Change runtime type). Needs a Colab secret **`HF_TOKEN`** (key icon in the
left sidebar, notebook access on) holding a Hugging Face read token with access to FineVideo.

Results go to Google Drive: `MyDrive/alethia_run/`. Cells are safe to re-run; the full run resumes.

## 1. Machine info

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core'
!free -g | head -2
!df -h /content | tail -1
!python --version

## 2. Google Drive (results folder)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/alethia_run'
os.makedirs(OUT, exist_ok=True)
print('results ->', OUT)

## 3. Code

In [ ]:
%cd /content
!test -d Alethia-AI-Assignment || git clone https://github.com/Qasim-Sajjad/Alethia-AI-Assignment.git
%cd /content/Alethia-AI-Assignment
!git pull --ff-only
!git log --oneline -3

## 4. Install (requirements + TensorRT)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q tensorrt-cu12 onnx onnxslim huggingface_hub
!pip freeze | grep -iE '^(torch|torchvision|ultralytics|av|pyarrow|numpy|opencv-python|tensorrt|onnx|onnxslim|huggingface)' | tee $OUT/pip_freeze.txt

## 5. Download the first 50 FineVideo Parquet files (local disk, ~25 GB)

In [ ]:
from google.colab import userdata
import os, time
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import snapshot_download
t = time.time()
snapshot_download('HuggingFaceFV/finevideo', repo_type='dataset', allow_patterns=['data/train-000[0-4]*'],
                  local_dir='/content/finevideo', max_workers=16)
files = sorted(os.listdir('/content/finevideo/data'))
print(len(files), 'files', f'{time.time() - t:.0f}s')
!du -sh /content/finevideo/data
# 66-video sample (files 00000-00001) used for the reference comparison
os.makedirs('/content/sample2', exist_ok=True)
for f in files[:2]:
    dst = f'/content/sample2/{f}'
    if not os.path.exists(dst):
        os.symlink(f'/content/finevideo/data/{f}', dst)
print(sorted(os.listdir('/content/sample2')))

## 6. Preparation step (what the graders run once): weights + TensorRT engine

In [ ]:
import time
t = time.time()
!python prepare.py
print(f'prepare: {time.time() - t:.0f}s')
!ls -la weights/

## 7. Sample benchmarks (66 videos, 5.21 h) — scored against the every-frame FP32 reference
Reference cache: `eval/ref_66/` (built locally on the RTX 3050 with the baseline).

In [ ]:
# a) fast pipeline, PyTorch FP16
!python run.py --input /content/sample2 --output $OUT/sample_fp16.jsonl --weights weights/yolo11l.pt
!python tools/score.py --ref-dir eval/ref_66 --pred $OUT/sample_fp16.jsonl | head -20

In [ ]:
# b) fast pipeline, TensorRT FP16 engine (default when weights/yolo11l.engine exists), with per-frame dump
!python run.py --input /content/sample2 --output $OUT/sample_trt.jsonl --dump-dir /content/dump_trt
!python tools/score.py --ref-dir eval/ref_66 --pred $OUT/sample_trt.jsonl | head -20
!python tools/compare_dumps.py --ref-dir eval/ref_66 --dump-dir /content/dump_trt

In [ ]:
import json
for name in ['sample_fp16', 'sample_trt']:
    p = f'{OUT}/{name}.jsonl.timings.json'
    if os.path.exists(p):
        r = json.load(open(p))
        print(name, {k: r[k] for k in ['weights', 'wall_s', 'video_fps', 'inferred_pct', 'workers', 'batch', 'gpu_mean_batch']})
        print('   gpu', {k: round(v, 1) for k, v in r['gpu_s'].items()})
        print('   workers', {k: round(v, 1) for k, v in r['workers_sum_s'].items()})

## 8. Full run: all 50 files → Drive (resumes if interrupted)

In [ ]:
!python run.py --input /content/finevideo/data --output $OUT/results.jsonl --resume
!wc -l $OUT/results.jsonl
!grep -c '"error"' $OUT/results.jsonl || true
!gzip -kf $OUT/results.jsonl && ls -la $OUT/
# sanity: the 66 sample videos inside the full run, scored against the reference
!python tools/score.py --ref-dir eval/ref_66 --pred $OUT/results.jsonl | head -20

## 9. Hardware record for HARDWARE.md

In [ ]:
!(nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv; lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core'; free -g | head -2; df -h /content | tail -1; python --version; cat /etc/os-release | head -2) | tee $OUT/hardware.txt
!cat $OUT/results.jsonl.timings.json

## 10. (Optional, ~1 h) Baseline on the same 66 videos for the first-vs-final comparison
Every frame, single process, FP32 — the first working version.

In [ ]:
import time
t = time.time()
!python tools/baseline_run.py --input /content/sample2 --output $OUT/sample_baseline.jsonl | tail -2
print(f'baseline wall: {time.time() - t:.0f}s')